# 12 — Sensitivity Analysis and Robustness
**References:** Cornfield et al. (1959) · Rosenbaum (2002) *Observational Studies* · VanderWeele & Ding (2017, *Annals of Internal Medicine*) · Cinelli & Hazlett (2020, *JRSS-B*) · Hernán & Robins (2020) Ch. 7

## Narrative thread
```
Unconfoundedness is untestable -> "How strong would a confounder have to be?" -> E-values -> Rosenbaum-style bounds -> Placebo & negative controls
```

## The question every observational study must answer

Notebooks 05–06 *assumed* no unmeasured confounding. That assumption is untestable, so the
honest endgame is quantitative: **how strong would an unmeasured confounder need to be to
explain the result away?**

The template is Cornfield et al. (1959) on smoking and lung cancer: for a hidden factor to
produce the observed risk ratio of ~9, it would have to be at least 9× more prevalent among
smokers *and* a huge cause of cancer — implausible, and the argument that settled the
debate long before any RCT could.

## The E-value (VanderWeele & Ding 2017)

For an observed risk ratio $RR > 1$:
$$E = RR + \sqrt{RR\,(RR - 1)}$$

is the minimum strength of association (risk-ratio scale) that an unmeasured confounder
would need with **both** treatment and outcome, above the measured covariates, to fully
explain the estimate. Report it for the point estimate and for the CI bound nearest the
null. Small E-value (≈1.3): fragile. Large (≈5+): sturdy.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import statsmodels.api as sm
import statsmodels.formula.api as smf

plt.rcParams.update({
    'figure.dpi': 130, 'axes.spines.top': False, 'axes.spines.right': False,
    'axes.grid': True, 'grid.color': '#e8e8e8', 'axes.axisbelow': True,
    'axes.titlesize': 11, 'axes.titleweight': 'bold', 'legend.frameon': False,
})
np.random.seed(42)

In [ ]:
# ── E-values: computing and displaying the bias curve ────────────────────
def evalue(rr):
    rr = max(rr, 1/rr)                     # symmetric for protective effects
    return rr + np.sqrt(rr * (rr - 1))

examples = {
    'Smoking -> lung cancer (Cornfield): RR ~ 9': 9.0,
    'Typical epi finding: RR = 1.5': 1.5,
    'Weak association: RR = 1.15': 1.15,
}
for name, rr in examples.items():
    print(f'{name:<48} E-value = {evalue(rr):.2f}')

# The joint-strength frontier that exactly explains away RR_obs:
# RR_UD * RR_WU / (RR_UD + RR_WU - 1) = RR_obs  (Ding & VanderWeele 2016)
fig, ax = plt.subplots(figsize=(7.5, 4.5))
for rr_obs, color in [(1.5, '#1f77b4'), (2.5, '#2ca02c'), (9.0, '#d62728')]:
    rr_wu = np.linspace(rr_obs + 0.01, 20, 300)
    rr_ud = rr_obs * (rr_wu - 1) / (rr_wu - rr_obs)
    ok = rr_ud > 0
    ax.plot(rr_wu[ok], rr_ud[ok], color=color, lw=2, label=f'explains away RR = {rr_obs}')
    e = evalue(rr_obs)
    ax.plot(e, e, 'o', color=color, ms=7)
ax.set_xlim(1, 20); ax.set_ylim(1, 20)
ax.set_xlabel('confounder -> treatment association (RR)')
ax.set_ylabel('confounder -> outcome association (RR)')
ax.set_title('Confounder strength needed to nullify an estimate\n(dot = E-value, the equal-strength point)')
ax.legend(); plt.tight_layout(); plt.show()

## Simulating the sensitivity question directly

The regression analogue (Cinelli & Hazlett 2020 formalize this as *robustness values*):
posit an unobserved confounder $U$ with given strengths on treatment and outcome, and
trace how the adjusted estimate moves. This turns "no unmeasured confounding" from a
binary leap of faith into a quantitative negotiation with skeptics.


In [ ]:
# ── Bias contour: how a hidden U of given strength moves the estimate ────
# True model: no effect at all (tau = 0); U drives both W and Y.
def biased_estimate(alpha_wu, beta_yu, n=200_000, seed=0):
    rng = np.random.default_rng(seed)
    u = rng.normal(0, 1, n)
    x = rng.normal(0, 1, n)                                  # measured covariate
    w = rng.binomial(1, 1/(1 + np.exp(-(0.5*x + alpha_wu*u))))
    y = 1.0*x + beta_yu*u + rng.normal(0, 1, n)              # tau = 0 !
    df = pd.DataFrame(dict(y=y, w=w, x=x))
    return smf.ols('y ~ w + x', df).fit().params['w']        # adjusts for x, not u

alphas = np.linspace(0, 1.5, 7)
betas  = np.linspace(0, 1.5, 7)
grid = np.array([[biased_estimate(a, b) for a in alphas] for b in betas])

fig, ax = plt.subplots(figsize=(7, 5))
cs = ax.contourf(alphas, betas, grid, levels=12, cmap='Reds')
ax.contour(alphas, betas, grid, levels=[0.1, 0.2, 0.3], colors='k', linewidths=1)
plt.colorbar(cs, label='spurious estimate of tau (true = 0)')
ax.set_xlabel('U -> treatment strength'); ax.set_ylabel('U -> outcome strength')
ax.set_title('Sensitivity surface: bias created by an unmeasured confounder')
plt.tight_layout(); plt.show()
print('A reader can now ask: "is a confounder this strong plausible,')
print('given that the MEASURED covariates have strengths of about 0.5 and 1.0?"')

## Rosenbaum's design sensitivity (matched studies)

For matched pairs, Rosenbaum (2002) bounds inference under hidden bias: suppose two
matched units' odds of treatment differ by at most $\Gamma$
($\frac{1}{\Gamma} \leq \frac{\pi_i / (1-\pi_i)}{\pi_j/(1-\pi_j)} \leq \Gamma$).
For each $\Gamma \geq 1$, the randomization p-value becomes an interval; report
$\Gamma^*$, the largest $\Gamma$ at which the result stays significant. $\Gamma^* = 1.1$
is fragile; smoking-and-lung-cancer survives $\Gamma \approx 6$.

## Placebo and negative-control checks

Cheap, powerful falsification tests that every observational paper should run:

| Check | Logic | Example |
|---|---|---|
| Negative-control outcome | Treatment can't affect it ⟹ estimated "effect" = confounding | Flu shot vs *non-flu* mortality (healthy-user bias) |
| Negative-control exposure | Fake treatment shouldn't move the outcome | Sibling's treatment status |
| Placebo timing | Effect appears *before* treatment ⟹ trend confounding | Pre-period "effects" in DiD (notebook 09) |
| Placebo cutoff / space | Jumps at fake thresholds / untreated units | RD (08), synthetic control (10) |


In [ ]:
# ── Negative-control outcome detects healthy-user confounding ────────────
# "Health-consciousness" u drives BOTH vaccination and general health.
n = 300_000
u = np.random.normal(0, 1, n)                        # unobserved health-consciousness
vaccinated = np.random.binomial(1, 1/(1 + np.exp(-1.5*u)))
flu_death     = np.random.binomial(1, np.clip(0.05 - 0.02*vaccinated + 0.015*(-u), 0.001, 1))
car_accident  = np.random.binomial(1, np.clip(0.05                   + 0.015*(-u), 0.001, 1))

df = pd.DataFrame(dict(v=vaccinated, flu=flu_death, car=car_accident))
rr_flu = df.query('v==1').flu.mean() / df.query('v==0').flu.mean()
rr_car = df.query('v==1').car.mean() / df.query('v==0').car.mean()
print(f'RR of flu death, vaccinated vs not:            {rr_flu:.2f}  (true effect exists)')
print(f'RR of CAR ACCIDENT death, vaccinated vs not:   {rr_car:.2f}  (true effect = none!)')
print('\nThe "protective effect" on an outcome the vaccine cannot touch exposes')
print('healthy-user confounding — and calibrates how much of the flu RR is real.')

## A closing checklist for any causal claim

1. **Draw the DAG** — make the identification assumptions explicit (03–04).
2. **Prefer design over adjustment** — RCT > quasi-experiment > selection-on-observables.
3. **Check overlap and balance** before fitting anything fancy (05–06).
4. **Match estimator to estimand** — ATE, ATT, LATE, effect-at-cutoff are different numbers.
5. **Stress-test**: sensitivity analysis, placebos, negative controls (this notebook).
6. **Report honestly** — the assumptions, the complier population, the E-value.

## Key takeaways

| Concept | Statement |
|---|---|
| Untestability | No unmeasured confounding cannot be verified from data |
| Cornfield logic | Bound the confounder strength needed to explain the result |
| E-value | $RR + \sqrt{RR(RR-1)}$ — a one-number sensitivity summary |
| Rosenbaum $\Gamma$ | Hidden-bias bounds for matched designs |
| Negative controls | Effects that *should* be zero measure your confounding |
| Robust practice | Design first, estimate second, stress-test always |
